# Phase 4 — Statistical Analysis & Hypothesis Testing
### SAS Hackathon Round 2: InsightPath / RUSTY WOLVES

This notebook executes the formal inferential statistical layer for the four data-science talent ecosystem cohorts.
It formally evaluates pre-registered hypotheses **H1 through H6** under pre-registered criteria:
- Non-parametric testing where distributional assumptions are violated
- Benjamini-Hochberg False Discovery Rate (FDR) multiple-testing control
- Dual reporting: Statistical significance ($p$-value, $q$-value) + Substantive effect sizes (Cohen's $d$, $r_{\text{rb}}$, Odds Ratios, Cramér's $V$) with 95% CIs
- Strict zero target leakage in predictive specifications
- Documented sensitivity analyses on anomaly-free sub-cohorts


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm

# Add project root to sys.path
sys.path.append(str(Path(".").resolve().parent.parent))

from src.statistics.assumptions import run_full_assumption_diagnostics
from src.statistics.group_tests import run_jds_skill_group_tests, run_sds_trait_group_tests
from src.statistics.correlations import run_h5_correlation_tests, run_h5_bivariate_regressions, run_h5_adjusted_regressions
from src.statistics.logistic_models import run_h2_jds_logistic, run_h4_sds_logistic, run_h6_analytics_logistic
from src.statistics.contingency import run_geographic_chisquare_test, run_regional_salary_rank_posthoc, run_skill_premium_associations
from src.statistics.sensitivity import run_h1_sensitivity_comparison, run_h2_sensitivity_comparison, run_h3_sensitivity_comparison, run_h4_sensitivity_comparison

print("Libraries and statistical modules successfully imported.")


## 1. Data Ingestion & Cohort Setup
Load the Phase 2 cleaned analytical datasets across the four separate cohorts.

In [ ]:
data_dir = Path("../../data/processed")
df_ds = pd.read_csv(data_dir / "data_science_jobs_processed.csv")
df_aj = pd.read_csv(data_dir / "analytics_jobs_processed.csv")
df_jds = pd.read_csv(data_dir / "jds_processed.csv")
df_jds_sens = pd.read_csv(data_dir / "jds_sensitivity_3291_removed.csv")
df_sds = pd.read_csv(data_dir / "sds_processed.csv")
df_sds_sens = df_sds.drop_duplicates(subset=["id"], keep="first").copy()

print(f"DataScience Jobs: {df_ds.shape}")
print(f"Analytics Jobs: {df_aj.shape}")
print(f"JDS Primary (N=139) | Sensitivity (N=137): {df_jds.shape}")
print(f"SDS Primary (N=161) | Deduplicated (N=152): {df_sds.shape}")


## 2. Distributional Assumption Diagnostics
Check normality via Shapiro-Wilk tests, skewness, kurtosis, and ceiling percentages.

In [ ]:
df_assumptions = run_full_assumption_diagnostics(df_jds, df_sds, df_ds, df_aj)
df_assumptions[["dataset", "variable", "class_group", "n", "skewness", "shapiro_p", "normality_violated", "ceiling_pct", "recommended_method"]].head(10)


## 3. Hypothesis 1 — Junior Technical Skills vs Salary Hike
Assess whether technical skills differentiate high salary hike recipients ($N=139$).

In [ ]:
h1_table = pd.read_csv("../../outputs/tables/phase4/phase4_h1_jds_group_tests.csv")
h1_fdr = pd.read_csv("../../outputs/tables/phase4/phase4_h1_fdr_results.csv")
h1_sens = pd.read_csv("../../outputs/tables/phase4/phase4_h1_sensitivity.csv")

display(h1_table[["variable", "mean_group1", "mean_group2", "mean_diff", "mann_whitney_u", "p_value_mwu", "cohens_d", "rank_biserial_r"]])
display(h1_fdr)
display(h1_sens[["variable", "primary_cohens_d", "sens_cohens_d", "primary_fdr_q", "sens_fdr_q", "stability_status"]])


## 4. Hypothesis 2 — Independent Junior Skill Associations
Multivariable binary logistic regression on standardized continuous skills.

In [ ]:
h2_table = pd.read_csv("../../outputs/tables/phase4/phase4_h2_jds_logistic.csv")
h2_vif = pd.read_csv("../../outputs/tables/phase4/phase4_h2_vif.csv")
h2_sens = pd.read_csv("../../outputs/tables/phase4/phase4_h2_sensitivity.csv")

display(h2_table[["parameter", "coef_beta", "std_error", "wald_z", "p_value", "odds_ratio", "or_ci_lower", "or_ci_upper", "sig_indicator"]])
display(h2_vif)
display(h2_sens[["predictor", "primary_or", "sens_or", "primary_p_val", "sens_p_val", "stability_status"]])


## 5. Hypothesis 3 — Senior Personality Trait Differences
Two-sample Mann-Whitney U tests comparing Big Five distributions across consulting success.

In [ ]:
h3_table = pd.read_csv("../../outputs/tables/phase4/phase4_h3_sds_group_tests.csv")
h3_fdr = pd.read_csv("../../outputs/tables/phase4/phase4_h3_fdr_results.csv")
h3_sens = pd.read_csv("../../outputs/tables/phase4/phase4_h3_sensitivity.csv")

display(h3_table[["variable", "mean_group1", "mean_group2", "mean_diff", "mann_whitney_u", "p_value_mwu", "cohens_d", "rank_biserial_r"]])
display(h3_fdr)
display(h3_sens[["trait", "primary_cohens_d", "dedup_cohens_d", "primary_fdr_q", "dedup_fdr_q", "stability_status"]])


## 6. Hypothesis 4 — Senior Personality Associations
Multivariable binary logistic regression on standardized traits.

In [ ]:
h4_table = pd.read_csv("../../outputs/tables/phase4/phase4_h4_sds_logistic.csv")
h4_vif = pd.read_csv("../../outputs/tables/phase4/phase4_h4_vif.csv")
h4_sens = pd.read_csv("../../outputs/tables/phase4/phase4_h4_sensitivity.csv")

display(h4_table[["parameter", "coef_beta", "std_error", "wald_z", "p_value", "odds_ratio", "or_ci_lower", "or_ci_upper", "sig_indicator"]])
display(h4_vif)
display(h4_sens[["predictor", "primary_or", "dedup_or", "primary_p_val", "dedup_p_val", "stability_status"]])


## 7. Hypothesis 5 — Experience-to-Compensation Elasticity
Bivariate correlation, OLS linear regression, semi-log regression, and title-adjusted models.

In [ ]:
h5_corr = pd.read_csv("../../outputs/tables/phase4/phase4_h5_correlation_tests.csv")
h5_reg = pd.read_csv("../../outputs/tables/phase4/phase4_h5_regression.csv")
h5_adj = pd.read_csv("../../outputs/tables/phase4/phase4_h5_adjusted_models.csv")

display(h5_corr)
display(h5_reg[["model_specification", "slope_beta", "slope_se_hc3", "ci_lower", "ci_upper", "p_value", "r_squared"]])
display(h5_adj[["model_specification", "experience_slope_beta", "experience_se_hc3", "ci_lower", "ci_upper", "p_value", "r_squared", "f_stat"]])


## 8. Hypothesis 6 — Premium Salary, Geography, and Technical Skills
Chi-square test of independence, regional rank comparisons, 2x2 skill contingency tables, and multivariable logistic regression.

In [ ]:
h6_geo = pd.read_csv("../../outputs/tables/phase4/phase4_h6_geography_chisquare.csv")
h6_skills = pd.read_csv("../../outputs/tables/phase4/phase4_h6_skill_associations.csv")
h6_log = pd.read_csv("../../outputs/tables/phase4/phase4_h6_logistic.csv")

display(h6_geo[["location_cluster", "total_postings", "high_salary_pct", "adj_standardized_residual_high", "local_direction"]])
display(h6_skills[["skill_name", "n_with_skill", "pct_high_with_skill", "odds_ratio", "ci_lower", "ci_upper", "fdr_adjusted_p", "significant_after_fdr"]].head(10))
display(h6_log[["parameter", "coef_beta", "std_error", "wald_z", "p_value", "odds_ratio", "or_ci_lower", "or_ci_upper", "sig_indicator"]])


## 9. Consolidated Hypothesis Decision Matrix
Consolidated decision matrix across all pre-registered hypotheses H1–H6.

In [ ]:
hypo_matrix = pd.read_csv("../../outputs/tables/phase4/phase4_hypothesis_summary.csv")
display(hypo_matrix[["hypothesis_id", "sample_size", "decision", "substantive_interpretation"]])
